# 🤖 NLP Intelligent Chatbot

## Project Overview
This notebook presents an intelligent chatbot powered by Natural Language Processing (NLP) techniques and state-of-the-art Large Language Models (LLMs). The chatbot is designed to understand user intent, analyze sentiment, recognize named entities, and provide context-aware responses. It leverages a hybrid approach, combining a custom-trained TensorFlow model for rapid intent classification with advanced generative AI models (Anthropic Claude and Google Gemini) for more nuanced and contextual interactions.

### Features:
-   **Intent Classification**: Recognizes user intentions to trigger scripted or generative responses.
-   **Sentiment Analysis**: Determines the emotional tone of user inputs for empathetic responses.
-   **Entity Recognition**: Identifies key information (like names, locations, dates) within user messages.
-   **Context-Aware Responses**: Utilizes conversation history and NLP insights to generate relevant replies.
-   **Dual AI Backend**: Supports both Anthropic Claude and Google Gemini for flexible generative capabilities.

---

This notebook builds a full NLP chatbot pipeline using:
-   **TensorFlow / Keras** — intent classification model (mirrors TensorFlow.js concepts)
-   **NLTK** — tokenization, lemmatization, NER
-   **TextBlob / VADER** — sentiment analysis
-   **spaCy** — named entity recognition
-   **Anthropic Claude API** — context-aware generative responses
-   **Google Gemini API** — alternative/fallback generative backend

---

## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install -q anthropic google-generativeai textblob vaderSentiment spacy nltk tensorflow
!python -m spacy download en_core_web_sm -q

import nltk
# Download necessary NLTK data for tokenization, lemmatization, and NER
nltk.download('punkt', quiet=True)       # For word and sentence tokenization
nltk.download('wordnet', quiet=True)    # For lemmatization
nltk.download('stopwords', quiet=True)  # For stop word removal (though not explicitly used in preprocessing here)
nltk.download('averaged_perceptron_tagger', quiet=True) # For part-of-speech tagging, used by some NLTK NER functions
nltk.download('maxent_ne_chunker', quiet=True)          # For NLTK's MaxEnt NE chunker
nltk.download('words', quiet=True)                      # For NLTK's word list
print('✅ All dependencies installed.')

## 🔑 Step 2 — API Keys
Add your keys to Colab Secrets (🔑 icon on the left panel) as `ANTHROPIC_API_KEY` and `GEMINI_API_KEY`, or paste them below.

In [ ]:
import os

# --- Option A: Colab Secrets (recommended) ---
try:
    from google.colab import userdata
    # Attempt to retrieve API keys from Colab's secret manager for secure storage.
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    GEMINI_API_KEY    = userdata.get('GEMINI_API_KEY')
    print('✅ Loaded keys from Colab Secrets.')
except Exception:
    # --- Option B: Paste directly ---
    # If Colab Secrets are not available or fail, fallback to hardcoded keys.
    # WARNING: Hardcoding keys is not recommended for production or shared notebooks.
    ANTHROPIC_API_KEY = 'YOUR_ANTHROPIC_KEY_HERE'
    GEMINI_API_KEY    = 'YOUR_GEMINI_KEY_HERE'
    print('⚠️  Using hardcoded keys — do not share this notebook.')

# Set API keys as environment variables for use by respective libraries.
os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY
os.environ['GEMINI_API_KEY']    = GEMINI_API_KEY

## 📚 Step 3 — Training Dataset (Intents)
Define intents with patterns and responses. This is the dataset the TensorFlow model will train on.

In [ ]:
import json

# Define the intents dataset for the chatbot.
# Each intent contains a 'tag' (category), 'patterns' (example user phrases),
# and 'responses' (predefined answers for that intent).
INTENTS = {
  "intents": [
    {
      "tag": "greeting",
      "patterns": ["hello", "hi", "hey", "good morning", "good evening", "howdy", "what's up", "sup"],
      "responses": ["Hello! How can I help you today?", "Hi there! What can I do for you?", "Hey! Great to see you."]
    },
    {
      "tag": "farewell",
      "patterns": ["bye", "goodbye", "see you", "take care", "later", "good night", "exit", "quit"],
      "responses": ["Goodbye! Have a great day!", "See you soon!", "Take care!"]
    },
    {
      "tag": "thanks",
      "patterns": ["thanks", "thank you", "thx", "cheers", "appreciate it", "that was helpful"],
      "responses": ["You're welcome!", "Happy to help!", "Anytime!"]
    },
    {
      "tag": "weather",
      "patterns": ["what is the weather", "weather today", "is it raining", "temperature outside", "forecast"],
      "responses": ["I don't have live weather data, but I can tell you about climate patterns!", "Check a weather app for real-time updates!"]
    },
    {\n      "tag": "name",
      "patterns": ["what is your name", "who are you", "what should I call you", "your name"],
      "responses": ["I'm NLPBot, your intelligent assistant!", "You can call me NLPBot."]
    },
    {
      "tag": "age",
      "patterns": ["how old are you", "what is your age", "when were you born", "your age"],
      "responses": ["I'm ageless — I exist in the cloud!", "Age is just a number for an AI like me!"]
    },
    {
      "tag": "help",
      "patterns": ["help", "I need help", "can you help", "assist me", "support"],
      "responses": ["Of course! Tell me what you need.", "I'm here to help. What's on your mind?"]
    },
    {
      "tag": "joke",
      "patterns": ["tell me a joke", "make me laugh", "say something funny", "joke"],
      "responses": ["Why did the AI go to school? To improve its neural network! 😄", "I told a joke about UDP but I wasn't sure if you'd get it."]
    },
    {
      "tag": "capabilities",
      "patterns": ["what can you do", "your abilities", "what are you capable of", "features", "functions"],
      "responses": ["I can chat, analyze sentiment, recognize entities, and answer questions using Claude and Gemini!"]
    },
    {
      "tag": "complaint",
      "patterns": ["this is bad", "you are useless", "terrible", "awful", "I hate this", "not helpful"],
      "responses": ["I'm sorry to hear that. Let me try to do better!", "I apologize. Can you tell me more so I can improve?"]
    }
  ]
}

print(f'✅ Dataset loaded: {len(INTENTS["intents"])} intents defined.')

## 🔧 Step 4 — NLP Preprocessing

In [ ]:
import numpy as np
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Initialize the WordNet Lemmatizer to reduce words to their base or dictionary form.
lemmatizer = WordNetLemmatizer()

def preprocess(sentence):
    """Tokenize, lowercase, and lemmatize a sentence, removing non-alphabetic tokens."""
    # Tokenize the sentence into individual words and convert to lowercase.
    tokens = word_tokenize(sentence.lower())
    # Lemmatize each token and filter out any tokens that are not purely alphabetic.
    return [lemmatizer.lemmatize(t) for t in tokens if t.isalpha()]

# --- Data Preparation for Model Training ---
# Initialize lists to store all unique words, all unique intent tags, and (pattern, tag) pairs.
all_words = []  # Stores all unique words from patterns to form the vocabulary.
tags = []       # Stores all unique intent tags.
xy = []         # Stores tuples of (processed_pattern_words, corresponding_tag).

# Iterate through each intent defined in the INTENTS dataset.
for intent in INTENTS['intents']:
    tag = intent['tag']
    tags.append(tag) # Add the intent tag to our list of tags.

    # Process each pattern associated with the current intent.
    for pattern in intent['patterns']:
        words = preprocess(pattern) # Preprocess the pattern sentence.
        all_words.extend(words)     # Add processed words to the overall word list.
        xy.append((words, tag))     # Store the word list and its corresponding tag.

# Create a sorted list of unique words (vocabulary) and unique tags.
all_words = sorted(set(all_words))
tags      = sorted(set(tags))

print(f'✅ Vocabulary size : {len(all_words)}') # Display the size of the vocabulary.
print(f'✅ Number of tags  : {len(tags)}')    # Display the number of unique intent tags.
print(f'✅ Training samples: {len(xy)}')    # Display the total number of (pattern, tag) pairs.

# --- Bag-of-Words (BoW) Encoding ---
def bag_of_words(sentence_words, vocab):
    """Converts a list of words into a bag-of-words vector based on the given vocabulary."""
    # Create a vector where each element is 1 if the word is present in sentence_words,
    # and 0 otherwise, corresponding to the position in the vocabulary.
    return np.array([1 if w in sentence_words else 0 for w in vocab], dtype=np.float32)

# Generate training data (X_train) and labels (y_train).
# X_train: Bag-of-words representation for each preprocessed pattern.
X_train = np.array([bag_of_words(words, all_words) for words, _ in xy])
# y_train: Numerical encoding of the intent tag for each pattern.
y_train = np.array([tags.index(tag) for _, tag in xy])

print(f'✅ X_train shape: {X_train.shape}') # Display the shape of the training features.
print(f'✅ y_train shape: {y_train.shape}') # Display the shape of the training labels.

## 🧠 Step 5 — Build & Train the TensorFlow Intent Classifier
This mirrors what TensorFlow.js does in the browser — we train the equivalent model in Python.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.utils import to_categorical

tf.random.set_seed(42)

y_cat = to_categorical(y_train, num_classes=len(tags))

# Model architecture
model = models.Sequential([
    layers.Input(shape=(len(all_words),)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(len(tags), activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

early_stop = callbacks.EarlyStopping(monitor='loss', patience=20, restore_best_weights=True)

history = model.fit(
    X_train, y_cat,
    epochs=300,
    batch_size=8,
    verbose=0,
    callbacks=[early_stop]
)

final_acc  = history.history['accuracy'][-1]
final_loss = history.history['loss'][-1]
print(f'✅ Training complete — Accuracy: {final_acc:.4f} | Loss: {final_loss:.4f}')

## 📊 Step 6 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'], color='steelblue', linewidth=2)
ax1.set_title('Model Accuracy', fontsize=14)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.grid(alpha=0.3)

ax2.plot(history.history['loss'], color='tomato', linewidth=2)
ax2.set_title('Model Loss', fontsize=14)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print('✅ Training curves plotted.')

## 😊 Step 7 — Sentiment Analysis (VADER + TextBlob)

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

vader = SentimentIntensityAnalyzer()

def analyze_sentiment(text):
    """
    Returns a dict with:
      - label     : 'positive' | 'negative' | 'neutral'
      - compound  : VADER compound score (-1 to 1)
      - polarity  : TextBlob polarity (-1 to 1)
      - subjectivity: TextBlob subjectivity (0 to 1)
    """
    scores    = vader.polarity_scores(text)
    compound  = scores['compound']
    blob      = TextBlob(text)

    if compound >= 0.05:
        label = 'positive'
    elif compound <= -0.05:
        label = 'negative'
    else:
        label = 'neutral'

    return {
        'label'        : label,
        'compound'     : round(compound, 4),
        'polarity'     : round(blob.sentiment.polarity, 4),
        'subjectivity' : round(blob.sentiment.subjectivity, 4)
    }

# Demo
test_sentences = [
    "I absolutely love this chatbot, it's amazing!",
    "This is terrible and completely useless.",
    "The weather is okay I guess."
]

print('=== Sentiment Analysis Demo ===\n')
for s in test_sentences:
    result = analyze_sentiment(s)
    print(f'Text      : {s}')
    print(f'Sentiment : {result}\n')

## 🏷️ Step 8 — Named Entity Recognition (spaCy)

In [ ]:
import spacy

nlp_spacy = spacy.load('en_core_web_sm')

def extract_entities(text):
    """
    Returns a list of (entity_text, entity_label) tuples.
    Common labels: PERSON, ORG, GPE, DATE, MONEY, PRODUCT, etc.
    """
    doc = nlp_spacy(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

# Demo
entity_tests = [
    "Elon Musk founded SpaceX in California in 2002.",
    "I want to book a flight to Lagos on Friday for $200.",
    "Google and Microsoft are competing in the AI space."
]

print('=== Named Entity Recognition Demo ===\n')
for text in entity_tests:
    entities = extract_entities(text)
    print(f'Text     : {text}')
    print(f'Entities : {entities}\n')

## 🔮 Step 9 — Intent Prediction

In [ ]:
import random

CONFIDENCE_THRESHOLD = 0.6

def predict_intent(user_input):
    """
    Returns (tag, confidence, scripted_response).
    If confidence is low, tag = 'unknown'.
    """
    words   = preprocess(user_input)
    bow     = bag_of_words(words, all_words).reshape(1, -1)
    probs   = model.predict(bow, verbose=0)[0]
    idx     = np.argmax(probs)
    confidence = float(probs[idx])

    if confidence < CONFIDENCE_THRESHOLD:
        return 'unknown', confidence, None

    tag = tags[idx]
    # Find scripted response
    for intent in INTENTS['intents']:
        if intent['tag'] == tag:
            return tag, confidence, random.choice(intent['responses'])

    return tag, confidence, None

# Demo
test_inputs = ['hey there', 'tell me a joke', 'what can you do', 'I am very sad today']
print('=== Intent Prediction Demo ===\n')
for inp in test_inputs:
    tag, conf, resp = predict_intent(inp)
    print(f'Input     : "{inp}"')
    print(f'Intent    : {tag} ({conf:.2%})')
    print(f'Response  : {resp}\n')

## 🤖 Step 10 — Claude API (Context-Aware Responses)

In [ ]:
import anthropic

claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def ask_claude(user_input, conversation_history, sentiment, entities):
    """
    Sends the user message + context (sentiment, entities, history) to Claude.
    Returns the assistant reply string.
    """
    system_prompt = f"""You are NLPBot, an intelligent, friendly, and context-aware chatbot.
You are equipped with NLP capabilities including sentiment analysis and entity recognition.

Use the following context to tailor your response:
- User's detected sentiment: {sentiment['label']} (score: {sentiment['compound']})
- Entities detected in their message: {entities if entities else 'none'}

If the user seems negative or upset, be empathetic and supportive.
If they seem positive, match their energy.
Keep responses concise (2-4 sentences) unless a detailed answer is genuinely needed."""

    messages = conversation_history + [{"role": "user", "content": user_input}]

    response = claude_client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=512,
        system=system_prompt,
        messages=messages
    )
    return response.content[0].text

print('✅ Claude client ready.')

## ✨ Step 11 — Gemini API (Alternative Backend)

In [ ]:
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel('gemini-1.5-flash')

def ask_gemini(user_input, conversation_history, sentiment, entities):
    """
    Sends the user message + NLP context to Gemini.
    Returns the reply string.
    """
    context_block = f"""
[SYSTEM] You are NLPBot, a friendly context-aware chatbot.
Detected sentiment: {sentiment['label']} (score {sentiment['compound']}).
Entities in message: {entities if entities else 'none'}.
Adjust tone accordingly. Keep reply under 4 sentences.
"""
    # Build history string
    history_str = '\n'.join(
        [f"{m['role'].upper()}: {m['content']}" for m in conversation_history[-6:]]
    )
    prompt = f"{context_block}\n{history_str}\nUSER: {user_input}\nNLPBOT:"

    response = gemini_model.generate_content(prompt)
    return response.text.strip()

print('✅ Gemini client ready.')

## 🧩 Step 12 — Full Chatbot Pipeline

In [ ]:
def chatbot_respond(user_input, conversation_history, backend='claude'):
    """
    Full pipeline:
      1. Sentiment analysis
      2. Entity recognition
      3. Intent classification (TF model)
      4. If high-confidence intent → scripted response
         Else → generative response via Claude or Gemini
      5. Update conversation history

    Returns: (reply, sentiment, entities, intent_tag, confidence)
    """
    # --- NLP Analysis ---
    sentiment = analyze_sentiment(user_input)
    entities  = extract_entities(user_input)
    tag, conf, scripted = predict_intent(user_input)

    # --- Response selection ---
    if scripted and conf >= CONFIDENCE_THRESHOLD:
        reply = scripted
        source = 'scripted'
    else:
        # Use generative AI for complex / unknown inputs
        try:
            if backend == 'gemini':
                reply = ask_gemini(user_input, conversation_history, sentiment, entities)
                source = 'gemini'
            else:
                reply = ask_claude(user_input, conversation_history, sentiment, entities)
                source = 'claude'
        except Exception as e:
            reply  = f"⚠️ AI backend error: {e}"
            source = 'error'

    # --- Update history ---
    conversation_history.append({"role": "user",      "content": user_input})
    conversation_history.append({"role": "assistant", "content": reply})

    return reply, sentiment, entities, tag, conf, source

print('✅ Chatbot pipeline ready.')

## 🖥️ Step 13 — Interactive Chat Demo

In [ ]:
# ===== CONFIGURE HERE =====
BACKEND = 'claude'   # <-- change to 'gemini' to use Gemini instead
# ==========================

conversation_history = []

print('='*60)
print('  🤖  NLP CHATBOT  (type "quit" to exit)')
print(f'  Backend: {BACKEND.upper()}')
print('='*60)

while True:
    user_input = input('\nYou: ').strip()
    if not user_input:
        continue
    if user_input.lower() in ['quit', 'exit', 'bye']:
        print('Bot: Goodbye! 👋')
        break

    reply, sentiment, entities, tag, conf, source = chatbot_respond(
        user_input, conversation_history, backend=BACKEND
    )

    print(f'\nBot: {reply}')
    print(f'  📊 Sentiment : {sentiment["label"]} (compound={sentiment["compound"]})')
    print(f'  🏷️  Entities  : {entities if entities else "none"}')
    print(f'  🎯 Intent    : {tag} ({conf:.0%} confidence) | source: {source}')

## 🔄 Step 14 — Batch Evaluation

In [ ]:
import pandas as pd

test_queries = [
    "Hello there!",
    "What is your name?",
    "Tell me a joke.",
    "I need some help please.",
    "Elon Musk visited Nigeria last month.",
    "I am feeling really depressed and hopeless.",
    "What's the capital of France?",
    "Can you help me write a Python function?",
    "Goodbye, see you later!",
    "This chatbot is incredible, I love it!"
]

results = []
history = []

for q in test_queries:
    reply, sentiment, entities, tag, conf, source = chatbot_respond(q, history, backend='claude')
    results.append({
        'Input'      : q,
        'Intent'     : tag,
        'Confidence' : f'{conf:.0%}',
        'Sentiment'  : sentiment['label'],
        'Entities'   : str(entities) if entities else '-',
        'Source'     : source,
        'Response'   : reply[:80] + '...' if len(reply) > 80 else reply
    })

df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', 60)
print('=== Batch Evaluation Results ===')
display(df)

## 📈 Step 15 — Visualization Dashboard

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

sentiments = [r['Sentiment'] for r in results]
intents    = [r['Intent']    for r in results]
sources    = [r['Source']    for r in results]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('NLP Chatbot — Evaluation Dashboard', fontsize=16, fontweight='bold')

# Sentiment distribution
sent_counts = Counter(sentiments)
colors_sent = {'positive': '#4CAF50', 'neutral': '#2196F3', 'negative': '#F44336'}
axes[0].bar(sent_counts.keys(),
            sent_counts.values(),
            color=[colors_sent.get(k, 'gray') for k in sent_counts.keys()])
axes[0].set_title('Sentiment Distribution')
axes[0].set_ylabel('Count')

# Intent distribution
intent_counts = Counter(intents)
axes[1].barh(list(intent_counts.keys()), list(intent_counts.values()), color='steelblue')
axes[1].set_title('Intent Distribution')
axes[1].set_xlabel('Count')

# Response source
source_counts = Counter(sources)
axes[2].pie(source_counts.values(),
            labels=source_counts.keys(),
            autopct='%1.0f%%',
            colors=['#FF9800', '#9C27B0', '#607D8B'])
axes[2].set_title('Response Source')

plt.tight_layout()
plt.show()
print('✅ Dashboard rendered.')

## 💾 Step 16 — Save the Model

In [ ]:
import pickle

# Save TensorFlow model
model.save('nlp_chatbot_model.keras')

# Save vocabulary and tags
with open('chatbot_vocab.pkl', 'wb') as f:
    pickle.dump({'all_words': all_words, 'tags': tags, 'intents': INTENTS}, f)

print('✅ Model saved as nlp_chatbot_model.keras')
print('✅ Vocabulary saved as chatbot_vocab.pkl')

---
## ✅ Summary

| Component | Technology Used |
|---|---|
| Intent Classification | TensorFlow / Keras (Dense + Dropout) |
| NLP Preprocessing | NLTK (tokenization, lemmatization) |
| Sentiment Analysis | VADER + TextBlob |
| Entity Recognition | spaCy (en_core_web_sm) |
| Context-Aware Responses | Anthropic Claude API |
| Alternative Backend | Google Gemini API |
| Dataset | Custom intents JSON (10 categories) |

### How it works
1. User input is preprocessed (tokenized + lemmatized).
2. Sentiment is extracted via VADER + TextBlob.
3. Named entities are extracted via spaCy.
4. The TensorFlow model classifies the intent.
5. High-confidence intents use scripted responses; unknown or complex queries go to Claude or Gemini for context-aware generation.
6. The conversation history is maintained across turns for multi-turn context.

---
*NLP Chatbot Assignment | Built with TensorFlow · NLTK · spaCy · VADER · Anthropic Claude · Google Gemini*